<a href="https://colab.research.google.com/github/sankhaBA/nlp-tryout/blob/main/fine_tune_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Packages

In [1]:
# Install required packages
%pip install -q transformers datasets scikit-learn torch --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.8 MB/s eta 0:00:00


### GPU Check

In [2]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


### Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Upload Dataset

In [4]:
import shutil

drive_csv_path = "/content/drive/MyDrive/Final Year Research Project (FYP)/Implementations/Feedback/nlp_tryout/Dataset/nav_dataset.csv"
shutil.copy(drive_csv_path, "nav_dataset.csv")
print("Dataset ready!")

Dataset ready!


### Load and Split Dataset

In [5]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load your CSV
df = pd.read_csv("nav_dataset.csv")

# Split: 90% for training, 10% for validation (checking during training)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

Training examples: 218
Validation examples: 25


### Tokenize Data

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

def preprocess(examples):
    model_inputs = tokenizer(
        ["navigate: " + x for x in examples["input"]],
        max_length=128,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["target"],
        max_length=64,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/218 [00:00<?, ? examples/s]

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

### Model Loading and Configuration

In [7]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq

model = T5ForConditionalGeneration.from_pretrained("t5-small")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = TrainingArguments(
    output_dir="./nav_t5_model_v1",
    num_train_epochs=10,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Train the model

In [ ]:
trainer.train()

c:\Users\SankhaAmbeypitiya\Projects\Other\fyp\feedback\nlp-tryout\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,4.722272,4.131915
2,4.318909,2.416339
3,2.617744,1.485033
4,2.082737,0.985436
5,1.199543,0.693906
6,0.955600,0.484518
7,0.842211,0.391505
8,0.695289,0.343140
9,0.566554,0.311995
10,0.585572,0.301915


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.90it/s]
c:\Users\SankhaAmbeypitiya\Projects\Other\fyp\feedback\nlp-tryout\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.99it/s]
c:\Users\SankhaAmbeypitiya\Projects\Other\fyp\feedback\nlp-tryout\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.39it/s]
c:\Users\SankhaAmbeypitiya\Projects\Other\fyp\feedback\nlp-tryout\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super

TrainOutput(global_step=140, training_loss=1.7907528843198504, metrics={'train_runtime': 165.7462, 'train_samples_per_second': 13.153, 'train_steps_per_second': 0.845, 'total_flos': 10892900302848.0, 'train_loss': 1.7907528843198504, 'epoch': 10.0})

### Save Model

In [ ]:
save_path = "/content/drive/MyDrive/Final Year Research Project (FYP)/Implementations/Feedback/nlp_tryout/models/nav_t5_final_v1"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.14it/s]

Model saved!


### Test Model

In [ ]:
### Test the Fine-Tuned Model
from transformers import T5ForConditionalGeneration, AutoTokenizer

model_path = "./nav_t5_final_v1"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)
model = model.to(device)
model.eval()

def generate_instruction(nav_json: dict) -> str:
    parts = [f"{k}: {v}" for k, v in nav_json.items()]
    input_text = "navigate: " + " ".join(parts)
    inputs = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(
        **inputs,
        max_length=64,
        min_new_tokens=4,
        num_beams=4,
        early_stopping=True
    )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return result if result else "[empty output - retrain with corrected label masking]"

test_cases = [
    {"action": "turn", "direction": "left", "distance": "10m", "landmark": "reception desk"},
    {"action": "continue", "distance": "20m"},
    {"action": "arrive", "landmark": "platform 3"},
    {"action": "stop", "reason": "hazard", "description": "wet floor"},
    {"action": "turn", "direction": "right", "distance": "5m", "landmark": "main entrance"},
]

print("=" * 60)
for tc in test_cases:
    print(f"Input:  {tc}")
    print(f"Output: {generate_instruction(tc)}")
    print("-" * 60)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 4678.67it/s]


Input:  {'action': 'turn', 'direction': 'left', 'distance': '10m', 'landmark': 'reception desk'}
Output: In 10m, turn left towards the reception desk.
------------------------------------------------------------
Input:  {'action': 'continue', 'distance': '20m'}
Output: Continue for 20m.
------------------------------------------------------------
Input:  {'action': 'arrive', 'landmark': 'platform 3'}
Output: Arrive at the platform 3.
------------------------------------------------------------
Input:  {'action': 'stop', 'reason': 'hazard', 'description': 'wet floor'}
Output: Stop at the wet floor.
------------------------------------------------------------
Input:  {'action': 'turn', 'direction': 'right', 'distance': '5m', 'landmark': 'main entrance'}
Output: In 5m, turn right towards the main entrance.
------------------------------------------------------------


### Comparison Against Baseline

In [ ]:
def template_instruction(nav_json: dict) -> str:
    action = nav_json.get("action")
    if action == "turn":
        return f"In {nav_json['distance']}, turn {nav_json['direction']} towards the {nav_json.get('landmark', 'next point')}."
    elif action == "continue":
        return f"Continue straight for {nav_json['distance']}."
    elif action == "arrive":
        return f"You have arrived at the {nav_json['landmark']}."
    elif action == "stop":
        return f"Stop. {nav_json.get('description', 'Obstacle')} ahead."
    return "Proceed as directed."

print("\nCOMPARISON: Fine-tuned model vs Template")
print("=" * 60)
for tc in test_cases:
    print(f"Input:    {tc}")
    print(f"Template: {template_instruction(tc)}")
    print(f"T5 Model: {generate_instruction(tc)}")
    print("-" * 60)

print("\nSUMMARY")
print("=" * 60)
matches = 0
for tc in test_cases:
    t = template_instruction(tc)
    m = generate_instruction(tc)
    match = "SAME" if t.strip().lower() == m.strip().lower() else "DIFF"
    if match == "SAME":
        matches += 1
    print(f"[{match}] {tc['action']}")
print(f"\nExact matches: {matches}/{len(test_cases)}")


COMPARISON: Fine-tuned model vs Template
Input:    {'action': 'turn', 'direction': 'left', 'distance': '10m', 'landmark': 'reception desk'}
Template: In 10m, turn left towards the reception desk.
T5 Model: In 10m, turn left towards the reception desk.
------------------------------------------------------------
Input:    {'action': 'continue', 'distance': '20m'}
Template: Continue straight for 20m.
T5 Model: Continue for 20m.
------------------------------------------------------------
Input:    {'action': 'arrive', 'landmark': 'platform 3'}
Template: You have arrived at the platform 3.
T5 Model: Arrive at the platform 3.
------------------------------------------------------------
Input:    {'action': 'stop', 'reason': 'hazard', 'description': 'wet floor'}
Template: Stop. wet floor ahead.
T5 Model: Stop at the wet floor.
------------------------------------------------------------
Input:    {'action': 'turn', 'direction': 'right', 'distance': '5m', 'landmark': 'main entrance'}
Templa